In [7]:
!python -m pip install numpy pandas scikit-learn joblib

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [2]:
TRAIN_PATH = Path("data/train-test.csv")
VALIDATION_PATH = Path("data/validation.csv")
DECEMBER_PATH = Path("data/december-chart-inputs.csv")

RANDOM_STATE = 42
TARGET = "posted_rate"
ID_COL = "load_id"

print("Setup complete")

Setup complete


In [3]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
december = pd.read_csv(DECEMBER_PATH)

print(f"Train shape: {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"December shape: {december.shape}")

Train shape: (48000, 14)
Validation shape: (12000, 13)
December shape: (31, 7)


In [4]:
train["date"] = pd.to_datetime(train["date"], errors="coerce")
validation["date"] = pd.to_datetime(validation["date"], errors="coerce")
december["date"] = pd.to_datetime(december["date"], errors="coerce")

print("Dates converted")

Dates converted


In [5]:
for col in ["distance", "weight", "market_index", "quote_signal", "posted_rate"]:
    if col in train.columns:
        train[col] = pd.to_numeric(train[col], errors="coerce")

for col in ["distance", "weight", "market_index", "quote_signal"]:
    if col in validation.columns:
        validation[col] = pd.to_numeric(validation[col], errors="coerce")

for col in ["distance", "weight"]:
    if col in december.columns:
        december[col] = pd.to_numeric(december[col], errors="coerce")

print("Numeric columns converted")

Numeric columns converted


In [6]:
for col in ["distance", "weight", "market_index", "quote_signal", "posted_rate"]:
    if col in train.columns:
        train.loc[train[col] < 0, col] = np.nan

for col in ["distance", "weight", "market_index", "quote_signal"]:
    if col in validation.columns:
        validation.loc[validation[col] < 0, col] = np.nan

for col in ["distance", "weight"]:
    if col in december.columns:
        december.loc[december[col] < 0, col] = np.nan

print("Negative values set to NaN")

Negative values set to NaN


In [14]:
train = train[train[TARGET].notna()].copy()
train = train[np.isfinite(train[TARGET])].copy()
train = train[train[TARGET] > 0].copy()

print(f"Train shape after target cleaning: {train.shape}")

Train shape after target cleaning: (48000, 14)


In [15]:
weight_global_median = train["weight"].median()
print(f"Global weight median: {weight_global_median}")

Global weight median: 31493.5


In [16]:
mask_weight_missing_train = train["weight"].isna()
train.loc[mask_weight_missing_train, "weight"] = weight_global_median

print(f"Filled {mask_weight_missing_train.sum()} missing weights in train")

Filled 592 missing weights in train


In [17]:
mask_weight_missing_val = validation["weight"].isna()
validation.loc[mask_weight_missing_val, "weight"] = weight_global_median

print(f"Filled {mask_weight_missing_val.sum()} missing weights in validation")

Filled 310 missing weights in validation


In [18]:
mask_weight_missing_dec = december["weight"].isna()
december.loc[mask_weight_missing_dec, "weight"] = weight_global_median

print(f"Filled {mask_weight_missing_dec.sum()} missing weights in december")

Filled 0 missing weights in december


In [19]:
market_index_by_date_train = train.groupby("date")["market_index"].median()

for idx, row in train.iterrows():
    if pd.isna(row["market_index"]):
        date = row["date"]
        if date in market_index_by_date_train.index:
            train.at[idx, "market_index"] = market_index_by_date_train[date]

print("Filled market_index in train using date median")

Filled market_index in train using date median


In [20]:
market_index_by_date_val = validation.groupby("date")["market_index"].median()

for idx, row in validation.iterrows():
    if pd.isna(row["market_index"]):
        date = row["date"]
        if date in market_index_by_date_val.index:
            validation.at[idx, "market_index"] = market_index_by_date_val[date]

print("Filled market_index in validation using date median")

Filled market_index in validation using date median


In [9]:
pickup_to_lat_lon = train.groupby("pickup")[["pickup_lat", "pickup_lon"]].first()
delivery_to_lat_lon = train.groupby("delivery")[["delivery_lat", "delivery_lon"]].first()

december["pickup_lat"] = december["pickup"].map(pickup_to_lat_lon["pickup_lat"])
december["pickup_lon"] = december["pickup"].map(pickup_to_lat_lon["pickup_lon"])
december["delivery_lat"] = december["delivery"].map(delivery_to_lat_lon["delivery_lat"])
december["delivery_lon"] = december["delivery"].map(delivery_to_lat_lon["delivery_lon"])

print("Added lat/lon to december from train")

Added lat/lon to december from train


In [10]:
market_index_by_date_dec = validation.groupby("date")["market_index"].median()
quote_signal_by_date_dec = validation.groupby("date")["quote_signal"].median()

december["market_index"] = december["date"].map(market_index_by_date_dec)
december["quote_signal"] = december["date"].map(quote_signal_by_date_dec)

print("Added market_index and quote_signal to december from validation")

Added market_index and quote_signal to december from validation


In [11]:
train["year"] = train["date"].dt.year
validation["year"] = validation["date"].dt.year
december["year"] = december["date"].dt.year

train["month"] = train["date"].dt.month
validation["month"] = validation["date"].dt.month
december["month"] = december["date"].dt.month

train["day"] = train["date"].dt.day
validation["day"] = validation["date"].dt.day
december["day"] = december["date"].dt.day

train["day_of_week"] = train["date"].dt.dayofweek
validation["day_of_week"] = validation["date"].dt.dayofweek
december["day_of_week"] = december["date"].dt.dayofweek

train["day_of_year"] = train["date"].dt.dayofyear
validation["day_of_year"] = validation["date"].dt.dayofyear
december["day_of_year"] = december["date"].dt.dayofyear

train["dow_sin"] = np.sin(2 * np.pi * train["day_of_week"] / 7)
train["dow_cos"] = np.cos(2 * np.pi * train["day_of_week"] / 7)
validation["dow_sin"] = np.sin(2 * np.pi * validation["day_of_week"] / 7)
validation["dow_cos"] = np.cos(2 * np.pi * validation["day_of_week"] / 7)
december["dow_sin"] = np.sin(2 * np.pi * december["day_of_week"] / 7)
december["dow_cos"] = np.cos(2 * np.pi * december["day_of_week"] / 7)

train["month_sin"] = np.sin(2 * np.pi * train["month"] / 12)
train["month_cos"] = np.cos(2 * np.pi * train["month"] / 12)
validation["month_sin"] = np.sin(2 * np.pi * validation["month"] / 12)
validation["month_cos"] = np.cos(2 * np.pi * validation["month"] / 12)
december["month_sin"] = np.sin(2 * np.pi * december["month"] / 12)
december["month_cos"] = np.cos(2 * np.pi * december["month"] / 12)

print("Added date features")

Added date features


In [24]:
train = train.sort_values("date").reset_index(drop=True)

split_index = int(len(train) * 0.80)

model_train = train.iloc[:split_index].copy()
holdout = train.iloc[split_index:].copy()

print(f"Model train: {len(model_train)} rows")
print(f"Holdout: {len(holdout)} rows")

Model train: 38400 rows
Holdout: 9600 rows


In [25]:
numeric_features = [
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

categorical_features = [
    "pickup",
    "delivery",
    "equipment",
]

In [26]:
X_train_numeric = model_train[numeric_features].copy()
X_train_categorical = model_train[categorical_features].copy()

scaler = StandardScaler()
X_train_numeric_scaled = scaler.fit_transform(X_train_numeric)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_categorical_encoded = encoder.fit_transform(X_train_categorical)

X_train_combined = np.hstack([X_train_numeric_scaled, X_train_categorical_encoded])
y_train = model_train[TARGET].values

print(f"X_train shape: {X_train_combined.shape}")

X_train shape: (38400, 144)


In [27]:
X_holdout_numeric = holdout[numeric_features].copy()
X_holdout_categorical = holdout[categorical_features].copy()

X_holdout_numeric_scaled = scaler.transform(X_holdout_numeric)
X_holdout_categorical_encoded = encoder.transform(X_holdout_categorical)

X_holdout_combined = np.hstack([X_holdout_numeric_scaled, X_holdout_categorical_encoded])
y_holdout = holdout[TARGET].values

print(f"X_holdout shape: {X_holdout_combined.shape}")

X_holdout shape: (9600, 144)


In [28]:
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 16),
    alpha=0.001,
    learning_rate_init=0.001,
    random_state=RANDOM_STATE,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=40
)

print("Training model...")
mlp.fit(X_train_combined, y_train)

print(f"Model iterations: {mlp.n_iter_}")

Training model...
Model iterations: 76


In [29]:
holdout_pred = mlp.predict(X_holdout_combined)
holdout_pred = np.maximum(holdout_pred, 0.01)

mae = mean_absolute_error(y_holdout, holdout_pred)
rmse = np.sqrt(mean_squared_error(y_holdout, holdout_pred))
r2 = r2_score(y_holdout, holdout_pred)
mape = np.mean(np.abs((y_holdout - holdout_pred) / y_holdout)) * 100

print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.6f}")
print(f"MAPE: {mape:.2f}%")

MAE:  137.535363
RMSE: 637.909963
R2:   0.825019
MAPE: 7.72%


In [30]:
X_final_numeric = train[numeric_features].copy()
X_final_categorical = train[categorical_features].copy()

final_scaler = StandardScaler()
X_final_numeric_scaled = final_scaler.fit_transform(X_final_numeric)

final_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_final_categorical_encoded = final_encoder.fit_transform(X_final_categorical)

X_final_combined = np.hstack([X_final_numeric_scaled, X_final_categorical_encoded])
y_final = train[TARGET].values

final_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 16),
    alpha=0.001,
    learning_rate_init=0.001,
    random_state=RANDOM_STATE,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=40
)

print("\nTraining final model...")
final_mlp.fit(X_final_combined, y_final)

print(f"Final model iterations: {final_mlp.n_iter_}")


Training final model...
Final model iterations: 71


In [31]:
joblib.dump(final_scaler, "final_scaler.joblib")
joblib.dump(final_encoder, "final_encoder.joblib")
joblib.dump(final_mlp, "final_mlp_model.joblib")

['final_mlp_model.joblib']

In [32]:
X_val_numeric = validation[numeric_features].copy()
X_val_categorical = validation[categorical_features].copy()

X_val_numeric_scaled = final_scaler.transform(X_val_numeric)
X_val_categorical_encoded = final_encoder.transform(X_val_categorical)

X_val_combined = np.hstack([X_val_numeric_scaled, X_val_categorical_encoded])

validation_pred = final_mlp.predict(X_val_combined)
validation_pred = np.maximum(validation_pred, 0.01)

validation_predictions_df = pd.DataFrame()
validation_predictions_df[ID_COL] = validation[ID_COL].values
validation_predictions_df["predicted_rate"] = validation_pred

val_pred_path = "validation_predictions.csv"
validation_predictions_df.to_csv(val_pred_path, index=False)

print(f"\nSaved validation predictions to {val_pred_path}")


Saved validation predictions to validation_predictions.csv


In [33]:
X_december_numeric = december[numeric_features].copy()
X_december_categorical = december[categorical_features].copy()

X_december_numeric_scaled = final_scaler.transform(X_december_numeric)
X_december_categorical_encoded = final_encoder.transform(X_december_categorical)

X_december_combined = np.hstack([X_december_numeric_scaled, X_december_categorical_encoded])

december_pred = final_mlp.predict(X_december_combined)
december_pred = np.maximum(december_pred, 0.01)

december["predicted_rate"] = december_pred

december_pred_path = "data/december_chart_inputs.csv"
december[["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]].to_csv(
    december_pred_path,
    index=False
)

print(f"Saved December predictions to {december_pred_path}")

print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.6f}")
print(f"MAPE: {mape:.2f}%")

Saved December predictions to data/december_chart_inputs.csv
MAE:  137.535363
RMSE: 637.909963
R2:   0.825019
MAPE: 7.72%


In [34]:
!python -m pip install -r requirements.txt
!python score.py --predictions validation_predictions.csv --december-predictions data/december_chart_inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results\candidate_december.png
Final validation metrics are calculated by Spotter after submission.
